## Wikipedia content

In [1]:
from langchain_community.document_loaders import WikipediaLoader

In [2]:
wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

## File-based content

In [3]:
from langchain_community.document_loaders import Docx2txtLoader

In [10]:
word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

In [11]:
from langchain_community.document_loaders import PyPDFLoader

In [12]:
pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

In [14]:
from langchain_community.document_loaders import TextLoader

In [15]:
txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

The document variables (`word_docs`, `pdf_docs`, `txt_docs`) are in **plural mode** because a loader always returns a list of documents, even if the list contains only one item.

## Creating the Document list

In [16]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

## Progressively refining the final summary

In [25]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import getpass

In [18]:
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [19]:
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENAI_API_KEY,
    model_name="gpt-5-nano"
)

In [20]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

In [21]:
doc_summary_chain = doc_summary_prompt | llm

In [22]:
refine_summary_template = """
You must produce a final summary from the current refined summary which
has been generated so far and from the content of an additional document.
This is the current refined summary generated so far:
{current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful,
otherwise return the current full summary as it is."""

In [23]:
refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

In [26]:
refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [30]:
def refine_summary(docs):
    intermediate_steps = []
    current_refined_summary = ''
    
    for doc in docs:
        intermediate_step = {
            "current_refined_summary": current_refined_summary,
            "text": doc.page_content
        }
        intermediate_steps.append(intermediate_step)
        current_refined_summary = refine_chain.invoke(intermediate_step)
    
    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [28]:
# full_summary = refine_summary(all_docs)